# Analysis of Netflix Movies and TV Shows

This notebook explores the Netflix titles dataset supplied for this project. We will prepare the data, study patterns in movies and TV shows, and use charts to answer questions about countries, ratings, genres, and release years.

The dataset contains titles available in a Netflix catalog snapshot. Its contents should not be treated as a live catalog.

## Data Preparation and Cleaning


We will import the libraries required for data analysis and visualization.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')

We will now load the CSV file into a pandas DataFrame. Keep the CSV in the same folder as this notebook.

In [ ]:
netflix_df = pd.read_csv('netflix_titles.csv')

Let us check the type and size of the dataset.

In [ ]:
type(netflix_df)

In [ ]:
netflix_df.shape

The dataset has rows for individual titles and columns describing each title.

In [ ]:
netflix_df.info()

Here are the column names included in the dataset.

In [ ]:
netflix_df.columns

We can inspect the first few records to understand the structure of the data.

In [ ]:
netflix_df.head(10)

The final few records help us confirm that the file was read through to the end.

In [ ]:
netflix_df.tail(10)

We will create a copy so that the original imported data remains available for comparison.

In [ ]:
netflix_df_copy = netflix_df.copy()

Let us count the missing values in each column before cleaning.

In [ ]:
netflix_df_copy.isnull().sum()

The dataset contains missing values in some descriptive columns. We will use explicit labels for missing country and rating values, and convert the date column to a date type for time-based analysis.

In [ ]:
netflix_df_copy['country'] = netflix_df_copy['country'].fillna('Unknown')
netflix_df_copy['rating'] = netflix_df_copy['rating'].fillna('Unknown')
netflix_df_copy['date_added'] = pd.to_datetime(netflix_df_copy['date_added'].str.strip(), errors='coerce')

We will remove leading and trailing spaces from text fields and check for duplicate rows.

In [ ]:
text_columns = netflix_df_copy.select_dtypes(include='object').columns
for column in text_columns:
    netflix_df_copy[column] = netflix_df_copy[column].str.strip()

netflix_df_copy.duplicated().sum()

No source rows will be removed without checking them. We can inspect any duplicate records below.

In [ ]:
netflix_df_copy[netflix_df_copy.duplicated(keep=False)].head(10)

The cleaned DataFrame retains useful descriptive columns such as director, cast, and description. We can review its final structure and remaining missing values.

In [ ]:
netflix_df_copy.info()

In [ ]:
netflix_df_copy.isnull().sum()

## Exploratory Analysis and Visualization


We will now visualize the data to better understand how the titles are distributed.

### Relation between Movies and TV Shows

In [ ]:
type_counts = netflix_df_copy['type'].value_counts()
ax = sns.countplot(data=netflix_df_copy, x='type', order=type_counts.index)
ax.set_title('Number of Movies and TV Shows')
ax.set_xlabel('Content type')
ax.set_ylabel('Number of titles')
plt.show()
type_counts

### Titles added to Netflix by year

In [ ]:
titles_by_added_year = netflix_df_copy.dropna(subset=['date_added']).assign(added_year=lambda df: df['date_added'].dt.year)['added_year'].value_counts().sort_index()
plt.figure(figsize=(12, 5))
sns.lineplot(x=titles_by_added_year.index, y=titles_by_added_year.values, marker='o')
plt.title('Netflix Titles Added by Year')
plt.xlabel('Year added')
plt.ylabel('Number of titles')
plt.xticks(rotation=45)
plt.show()
titles_by_added_year.tail(10)

### Ratings of Movies and Shows

In [ ]:
rating_counts = netflix_df_copy['rating'].value_counts().head(12)
plt.figure(figsize=(11, 6))
sns.barplot(x=rating_counts.values, y=rating_counts.index, color='steelblue')
plt.title('Most Common Content Ratings')
plt.xlabel('Number of titles')
plt.ylabel('Rating')
plt.show()
rating_counts

### Relation between Content Type and Rating

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=netflix_df_copy, y='rating', hue='type', order=netflix_df_copy['rating'].value_counts().index[:12])
plt.title('Ratings by Content Type')
plt.xlabel('Number of titles')
plt.ylabel('Rating')
plt.legend(title='Type')
plt.show()

### Release year distribution

In [ ]:
release_year_counts = netflix_df_copy['release_year'].value_counts().sort_index()
plt.figure(figsize=(13, 5))
plt.plot(release_year_counts.index, release_year_counts.values)
plt.title('Netflix Titles by Release Year')
plt.xlabel('Release year')
plt.ylabel('Number of titles')
plt.show()

### Movie duration

In [ ]:
movies = netflix_df_copy[netflix_df_copy['type'] == 'Movie'].copy()
movies['duration_minutes'] = pd.to_numeric(movies['duration'].str.extract(r'(\d+)')[0], errors='coerce')
movies['duration_minutes'].describe()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(movies['duration_minutes'].dropna(), bins=30, kde=True)
plt.title('Distribution of Movie Durations')
plt.xlabel('Duration in minutes')
plt.ylabel('Number of movies')
plt.show()

### TV show seasons

In [ ]:
tv_shows = netflix_df_copy[netflix_df_copy['type'] == 'TV Show'].copy()
tv_shows['seasons'] = pd.to_numeric(tv_shows['duration'].str.extract(r'(\d+)')[0], errors='coerce')
tv_shows['seasons'].value_counts().sort_index()

### Countries with the most titles

In [ ]:
country_counts = (netflix_df_copy[netflix_df_copy['country'] != 'Unknown']
                 .assign(country=netflix_df_copy['country'].str.split(', '))
                 .explode('country')['country'].value_counts().head(10))
plt.figure(figsize=(10, 6))
sns.barplot(x=country_counts.values, y=country_counts.index, color='teal')
plt.title('Top 10 Countries by Number of Titles')
plt.xlabel('Number of titles')
plt.ylabel('Country')
plt.show()
country_counts

### Most common genres

In [ ]:
genre_counts = (netflix_df_copy['listed_in'].dropna().str.split(', ').explode().value_counts().head(10))
plt.figure(figsize=(10, 6))
sns.barplot(x=genre_counts.values, y=genre_counts.index, color='darkorange')
plt.title('Top 10 Genres on Netflix')
plt.xlabel('Number of titles')
plt.ylabel('Genre')
plt.show()
genre_counts

## Asking and Answering Questions


### Q1: Does Netflix have more Movies or TV Shows?

In [ ]:
type_counts = netflix_df_copy['type'].value_counts()
type_counts

The values above show the number of titles in each content type. The larger value indicates which type is more common in this dataset.

### Q2: Which countries have the most titles?

In [ ]:
country_counts.head(5)

The table lists the five countries with the most titles. Titles with multiple countries are counted once for each listed country.

### Q3: What are the most common ratings?

In [ ]:
netflix_df_copy['rating'].value_counts().head(5)

This table shows the five most common ratings, including the Unknown category when present.

### Q4: What are the most common genres?

In [ ]:
genre_counts.head(5)

The five most common genres are based on splitting each title’s listed genres and counting each genre separately.

### Q5: Which release years have the most titles?

In [ ]:
netflix_df_copy['release_year'].value_counts().head(5)

The table ranks release years by the number of titles in this dataset.

### Q6: What is the typical movie duration?

In [ ]:
movies['duration_minutes'].describe()

The summary gives the count, average, spread, and quartiles for movie durations that could be parsed from the duration field.

### Q7: In which years were the most titles added to Netflix?

In [ ]:
titles_by_added_year.sort_values(ascending=False).head(5)

These years have the highest number of additions in the supplied snapshot. This reflects when titles were added to the catalog, not when they were originally released.

## Inferences and Conclusion


This analysis compares movies and TV shows and explores ratings, countries, genres, release years, catalog addition dates, and durations. Run the notebook from top to bottom to calculate the exact values and display the charts for the supplied CSV. The conclusions describe this dataset snapshot and should not be interpreted as current Netflix catalog statistics.

## References and Future Work

- Dataset: [Netflix Movies and TV Shows on Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows)
- Future work could compare catalog patterns across snapshots, analyze directors or cast, and investigate changes in genre and rating distributions over time.